# HLS Fmask Acceptance Test

## Purpose
This notebook confirms that a new HLS container correctly applies the validated Fmask algorithm.  
It compares `Fmask.tif` outputs from the candidate container against a **golden reference** — 
the scientifically validated Fmask outputs from the prior science review.

**This is not a new scientific validation of Fmask.** The science team has already validated 
Fmask5 improvements. This test simply confirms those improvements are correctly reproduced 
in the new container.

## Outline
1. **Configuration** — load parameters from `config/fmask_acceptance_config.yaml`
2. **Setup** — imports, AWS credentials, S3 client
3. **Data Discovery** — list Fmask files in candidate and golden reference buckets
4. **Curated Granule Tests** — compare known improvement cases against golden reference
5. **Random Sample Tests** — broader regression coverage against golden reference
6. **Pass / Fail Summary** — verdict per granule and overall

## How to run
- **Interactive:** Run all cells in JupyterLab. No notebook edits needed — update `config/fmask_acceptance_config.yaml` only.
- **Command line:** `python scripts/run_fmask_validation.py`

## Fmask Bit Encoding Reference
| Bit | Field | Values |
|-----|-------|--------|
| 0 | Cirrus (unused in HLS) | 0/1 |
| 1 | Cloud | 0=clear, 1=cloud |
| 2 | Adjacent to cloud | 0/1 |
| 3 | Cloud shadow | 0=clear, 1=shadow |
| 4 | Snow / Ice | 0=clear, 1=snow/ice |
| 5 | Water | 0=land, 1=water |
| 6–7 | Aerosol level | 0=climatology, 1=low, 2=moderate, 3=high |

---
## 1. Configuration
All parameters are loaded from the YAML config file.  
**Do not hardcode parameters below — edit the config file instead.**

In [ ]:
# ── Papermill parameter cell (tag: 'parameters') ──────────────────────────────
# When running via papermill, override config_path on the command line.
# Default resolves relative to the repo root.
import os
# notebook is in notebooks/, config is at ../config/ (one level up inside hls_validation_framework/)
config_path = os.path.join(os.path.dirname(os.path.abspath('')), '..', 'config', 'fmask_acceptance_config.yaml')

In [ ]:
import yaml

with open(config_path) as f:
    cfg = yaml.safe_load(f)

print(f"Fmask version under test : {cfg['fmask_version']}")
print(f"Golden reference bucket  : s3://{cfg['golden_reference_s3_bucket']}/{cfg['golden_reference_s3_prefix']}")
print(f"Candidate bucket         : s3://{cfg['candidate_s3_bucket']}/{cfg['candidate_s3_prefix']}")
print(f"Satellites               : {cfg['satellite_types']}")
print(f"Curated granules         : {len(cfg['curated_granules'])}")
print(f"Random sample size       : {cfg['random_sample']['size_per_satellite']} per satellite")
print(f"Pass thresholds          : curated={cfg['thresholds']['curated_pass_pct']}%  random={cfg['thresholds']['random_pass_pct']}%")

---
## 2. Setup

In [ ]:
import os
import random
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import boto3
import rasterio
from botocore.exceptions import NoCredentialsError, ClientError

# ── Load shared Fmask decoding functions ──────────────────────────────────────
%run -i ../module/fmask.ipynb
%run -i ../module/data_access.ipynb

# ── AWS client ────────────────────────────────────────────────────────────────
# Credentials must be set as environment variables — never hardcoded.
try:
    session = boto3.Session(region_name=cfg['aws_region'])
    creds   = session.get_credentials()
    assert creds is not None, "No AWS credentials found in environment."
    resolved = creds.get_frozen_credentials()
    s3 = session.client('s3', region_name=cfg['aws_region'])
    print(f"✅ AWS credentials loaded (key: {resolved.access_key[:8]}...)")
except (NoCredentialsError, AssertionError) as e:
    print("❌ No AWS credentials found. Set environment variables:")
    print("     export AWS_ACCESS_KEY_ID=...")
    print("     export AWS_SECRET_ACCESS_KEY=...")
    print("     export AWS_SESSION_TOKEN=...")
    raise

REPORT_DIR = os.path.join(os.path.dirname(os.path.abspath('')), '..', cfg['output']['report_dir'])
os.makedirs(REPORT_DIR, exist_ok=True)
print(f"📁 Report output dir: {os.path.abspath(REPORT_DIR)}")

---
## 3. Data Discovery
List Fmask.tif files in the golden reference and candidate buckets.

In [ ]:
def list_fmask_files(bucket, prefix):
    """List all Fmask.tif S3 keys under a given prefix."""
    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('Fmask.tif'):
                keys.append(obj['Key'])
    return sorted(keys)


golden_fmask_files   = list_fmask_files(
    cfg['golden_reference_s3_bucket'], cfg['golden_reference_s3_prefix'])

candidate_fmask_files = list_fmask_files(
    cfg['candidate_s3_bucket'], cfg['candidate_s3_prefix'])

print(f"Golden reference  : {len(golden_fmask_files)} Fmask files")
print(f"Candidate         : {len(candidate_fmask_files)} Fmask files")

# Extract granule IDs for matching
def granule_id(key):
    """Extract granule basename without extension from an S3 key."""
    fname = key.split('/')[-1]           # e.g. HLS.S30.T09VWL.2026059T200249.v2.0.Fmask.tif
    return fname.replace('.Fmask.tif', '')

golden_ids   = {granule_id(k): k for k in golden_fmask_files}
candidate_ids = {granule_id(k): k for k in candidate_fmask_files}

paired_ids = sorted(set(golden_ids) & set(candidate_ids))
print(f"Paired granules   : {len(paired_ids)} (present in both buckets)")

missing_in_candidate = set(golden_ids) - set(candidate_ids)
if missing_in_candidate:
    print(f"⚠️  {len(missing_in_candidate)} golden granules not found in candidate bucket")

---
## 4. Core Comparison Functions

In [ ]:
def read_fmask_from_s3(bucket, key):
    """Download Fmask.tif from S3 and return as uint8 numpy array."""
    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
        tmp_path = tmp.name
    try:
        s3.download_file(bucket, key, tmp_path)
        with rasterio.open(tmp_path) as src:
            return src.read(1).astype(np.uint8)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def compare_fmask(fmask_golden, fmask_candidate):
    """
    Bit-decode both Fmask arrays and compare per class.

    Returns a dict with per-class agreement rates and overall agreement.
    """
    # Decode each Fmask into class layers using existing module function
    masks_golden    = decode_hls_fmask(fmask_golden)
    masks_candidate = decode_hls_fmask(fmask_candidate)

    # Valid pixel mask (exclude fill value 255)
    valid = (fmask_golden != 255) & (fmask_candidate != 255)
    n_valid = int(valid.sum())

    results = {'n_valid_pixels': n_valid}

    class_keys = ['cloud', 'cloud_shadow', 'snow_ice', 'water', 'adjacent_cloud_shadow']
    all_agree = np.ones(fmask_golden.shape, dtype=bool)

    for cls in class_keys:
        g = masks_golden[cls]
        c = masks_candidate[cls]
        agree = (g == c)
        all_agree &= agree
        n_agree = int((agree & valid).sum())
        pct_agree = 100.0 * n_agree / n_valid if n_valid > 0 else float('nan')

        # Pixels that changed between golden and candidate
        n_changed = int(((~agree) & valid).sum())
        pct_changed = 100.0 * n_changed / n_valid if n_valid > 0 else float('nan')

        results[f'{cls}_agreement_pct'] = round(pct_agree, 4)
        results[f'{cls}_changed_pct']   = round(pct_changed, 4)

    # Aerosol level comparison
    aerosol_agree = (masks_golden['aerosol'] == masks_candidate['aerosol'])
    results['aerosol_agreement_pct'] = round(
        100.0 * int((aerosol_agree & valid).sum()) / n_valid if n_valid > 0 else float('nan'), 4)

    results['overall_agreement_pct'] = round(
        100.0 * int((all_agree & valid).sum()) / n_valid if n_valid > 0 else float('nan'), 4)

    return results


# Fmask class colormap for visualization
FMASK_CMAP_COLORS = {
    'clear':        (0.85, 0.85, 0.85, 1.0),
    'cloud':        (1.00, 1.00, 1.00, 1.0),
    'cloud_shadow': (0.30, 0.30, 0.30, 1.0),
    'snow_ice':     (0.53, 0.81, 0.98, 1.0),
    'water':        (0.13, 0.47, 0.71, 1.0),
    'fill':         (0.00, 0.00, 0.00, 1.0),
}


def fmask_to_class_array(fmask):
    """Convert raw Fmask uint8 array to a simplified 6-class integer array for display."""
    masks = decode_hls_fmask(fmask)
    classified = np.zeros(fmask.shape, dtype=np.uint8)  # 0 = clear
    classified[masks['water']]               = 1
    classified[masks['snow_ice']]            = 2
    classified[masks['cloud_shadow']]        = 3
    classified[masks['adjacent_cloud_shadow']] = 4
    classified[masks['cloud']]               = 5
    classified[fmask == 255]                 = 6  # fill
    return classified


def plot_fmask_comparison(fmask_golden, fmask_candidate, granule_id, save_path=None):
    """Side-by-side categorical Fmask maps + class change map."""
    import matplotlib.colors as mcolors

    class_labels = ['Clear', 'Water', 'Snow/Ice', 'Shadow', 'Adj.Shadow', 'Cloud', 'Fill']
    class_colors = ['#D9D9D9', '#2171B5', '#87CEEB', '#4D4D4D', '#969696', '#FFFFFF', '#000000']
    cmap = mcolors.ListedColormap(class_colors)
    norm = mcolors.BoundaryNorm(boundaries=list(range(len(class_labels)+1)), ncolors=len(class_labels))

    g_class = fmask_to_class_array(fmask_golden)
    c_class = fmask_to_class_array(fmask_candidate)
    change  = (g_class != c_class).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(granule_id, fontsize=11)

    for ax, data, title in zip(axes,
                               [g_class, c_class, None],
                               ['Golden Reference', 'Candidate', 'Changed Pixels']):
        if title == 'Changed Pixels':
            ax.imshow(change, cmap='Reds', vmin=0, vmax=1, interpolation='nearest')
            n_changed = int(change.sum())
            pct_changed = 100.0 * n_changed / change.size
            ax.set_title(f'{title}\n{n_changed:,} pixels changed ({pct_changed:.3f}%)', fontsize=9)
        else:
            ax.imshow(data, cmap=cmap, norm=norm, interpolation='nearest')
            ax.set_title(title, fontsize=9)

        ax.axis('off')

    patches = [mpatches.Patch(color=class_colors[i], label=class_labels[i])
               for i in range(len(class_labels))]
    fig.legend(handles=patches, loc='lower center', ncol=7,
               bbox_to_anchor=(0.5, -0.02), fontsize=8)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight', dpi=100)
    plt.show()
    plt.close(fig)


print("✅ Comparison functions defined")

---
## 5. Curated Granule Tests
These are the known improvement cases from the prior Fmask5 scientific validation.  
All curated granules **must** match the golden reference at the configured threshold.

In [ ]:
curated_results = []
PASS_PCT = cfg['thresholds']['curated_pass_pct']

for granule in cfg['curated_granules']:
    gid = granule['id']
    print(f"\n{'─'*70}")
    print(f"Granule    : {gid}")
    print(f"Scene type : {granule['scene_type']}")
    print(f"Notes      : {granule.get('notes', '')}")

    if gid not in golden_ids:
        print(f"  ⚠️  Not found in golden reference bucket — SKIPPED")
        curated_results.append({'granule_id': gid, 'status': 'SKIPPED',
                                 'reason': 'Not in golden reference'})
        continue

    if gid not in candidate_ids:
        print(f"  ❌ Not found in candidate bucket — FAIL")
        curated_results.append({'granule_id': gid, 'status': 'FAIL',
                                 'reason': 'Not in candidate bucket'})
        continue

    fmask_golden    = read_fmask_from_s3(cfg['golden_reference_s3_bucket'], golden_ids[gid])
    fmask_candidate = read_fmask_from_s3(cfg['candidate_s3_bucket'], candidate_ids[gid])

    stats = compare_fmask(fmask_golden, fmask_candidate)
    overall = stats['overall_agreement_pct']
    verdict = '✅ PASS' if overall >= PASS_PCT else '❌ FAIL'

    print(f"  Overall agreement : {overall:.4f}%  →  {verdict}")
    for cls in ['cloud', 'cloud_shadow', 'snow_ice', 'water']:
        print(f"    {cls:<22}: {stats[f'{cls}_agreement_pct']:.4f}% agreement  "
              f"({stats[f'{cls}_changed_pct']:.4f}% changed)")

    save_path = os.path.join(REPORT_DIR, 'curated', f'{gid}_fmask_comparison.png')
    plot_fmask_comparison(fmask_golden, fmask_candidate, gid, save_path=save_path)

    row = {'granule_id': gid, 'test_set': 'curated',
           'scene_type': granule['scene_type'],
           'overall_agreement_pct': overall,
           'status': 'PASS' if overall >= PASS_PCT else 'FAIL'}
    row.update(stats)
    curated_results.append(row)

df_curated = pd.DataFrame(curated_results)
print(f"\n{'='*70}")
print(f"Curated test set: {(df_curated['status']=='PASS').sum()} PASS  /  "
      f"{(df_curated['status']=='FAIL').sum()} FAIL  /  "
      f"{(df_curated['status']=='SKIPPED').sum()} SKIPPED")

---
## 6. Random Sample Tests
A random sample of paired granules for broader regression coverage.

In [ ]:
random_results = []
PASS_PCT_RAND = cfg['thresholds']['random_pass_pct']
SAMPLE_SIZE   = cfg['random_sample']['size_per_satellite']
SEED          = cfg['random_sample']['seed']

curated_ids_set = {g['id'] for g in cfg['curated_granules']}
pool = [gid for gid in paired_ids if gid not in curated_ids_set]

random.seed(SEED)
sample = random.sample(pool, min(SAMPLE_SIZE * len(cfg['satellite_types']), len(pool)))
print(f"Random sample size: {len(sample)} granules (seed={SEED})")

for gid in sample:
    print(f"  Testing: {gid} ... ", end='')
    try:
        fmask_golden    = read_fmask_from_s3(cfg['golden_reference_s3_bucket'], golden_ids[gid])
        fmask_candidate = read_fmask_from_s3(cfg['candidate_s3_bucket'], candidate_ids[gid])
        stats   = compare_fmask(fmask_golden, fmask_candidate)
        overall = stats['overall_agreement_pct']
        verdict = 'PASS' if overall >= PASS_PCT_RAND else 'FAIL'
        print(f"{overall:.4f}% agreement → {verdict}")
        row = {'granule_id': gid, 'test_set': 'random',
               'overall_agreement_pct': overall, 'status': verdict}
        row.update(stats)
        random_results.append(row)
    except Exception as e:
        print(f"ERROR: {e}")
        random_results.append({'granule_id': gid, 'test_set': 'random',
                                'status': 'ERROR', 'reason': str(e)})

df_random = pd.DataFrame(random_results)
print(f"\nRandom sample: {(df_random['status']=='PASS').sum()} PASS  /  "
      f"{(df_random['status']=='FAIL').sum()} FAIL  /  "
      f"{(df_random['status']=='ERROR').sum()} ERROR")

---
## 7. Pass / Fail Summary

In [ ]:
import datetime

df_all = pd.concat([df_curated, df_random], ignore_index=True)

n_curated_pass = (df_curated['status'] == 'PASS').sum()
n_curated_fail = (df_curated['status'] == 'FAIL').sum()
n_random_pass  = (df_random['status']  == 'PASS').sum()
n_random_fail  = (df_random['status']  == 'FAIL').sum()

overall_verdict = 'PASS' if (n_curated_fail == 0 and n_random_fail == 0) else 'FAIL'

print("="*70)
print(f"  HLS FMASK ACCEPTANCE TEST REPORT")
print(f"  Fmask version  : {cfg['fmask_version']}")
print(f"  Candidate      : s3://{cfg['candidate_s3_bucket']}/{cfg['candidate_s3_prefix']}")
print(f"  Run date       : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print(f"  Curated granules : {n_curated_pass} PASS  /  {n_curated_fail} FAIL  (threshold: {cfg['thresholds']['curated_pass_pct']}%)")
print(f"  Random sample    : {n_random_pass} PASS  /  {n_random_fail} FAIL  (threshold: {cfg['thresholds']['random_pass_pct']}%)")
print("-"*70)
print(f"  OVERALL VERDICT  : {'✅ PASS' if overall_verdict == 'PASS' else '❌ FAIL'}")
print("="*70)

if n_curated_fail > 0:
    print("\n⚠️  Failed curated granules:")
    failed = df_curated[df_curated['status'] == 'FAIL'][['granule_id', 'scene_type', 'overall_agreement_pct']]
    print(failed.to_string(index=False))

if n_random_fail > 0:
    print("\n⚠️  Failed random sample granules:")
    failed = df_random[df_random['status'] == 'FAIL'][['granule_id', 'overall_agreement_pct']]
    print(failed.to_string(index=False))

# Save full results CSV
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = os.path.join(REPORT_DIR, f'fmask_acceptance_{cfg["fmask_version"]}_{ts}.csv')
df_all.to_csv(csv_path, index=False)
print(f"\n📄 Full results saved to: {csv_path}")